# Cluster Skill Timelines

Colab notebook for building skill-share timelines from LinkedIn job postings. It clones the `vectorjobs` branch, reuses or rebuilds silver data, assigns each job to one of five fixed domains with Qwen embeddings, optionally enriches missing or uncertain skill rows with a GPT-OSS 20B model, persists DuckDB plus Parquet analytics, and writes one skill timeline plot per domain.

The temporal interpretation is conservative: if the available silver data only spans one or two months, the plots are comparisons across observed posting buckets, not stable long-term trend evidence.

## 0. Runtime and configuration

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except Exception:
    torch = None

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/fbientrigo/vectorjobs.git'
BRANCH = os.environ.get('VECTORJOBS_BRANCH', 'codex/temporal-drift-draft')
REPO_DIR = Path('/content/vectorjobs') if IN_COLAB else Path.cwd()

USE_DRIVE = IN_COLAB and os.environ.get('VECTORJOBS_USE_DRIVE', '1') == '1'
DRIVE_ROOT = Path('/content/drive/MyDrive/vectorjobs_skill_timelines') if IN_COLAB else REPO_DIR
PERSIST_ROOT = DRIVE_ROOT if USE_DRIVE else REPO_DIR

RAW_DIR = REPO_DIR / 'data/raw/linkedin-job-postings'
SILVER_DIR = REPO_DIR / 'data/silver'
SILVER_PATH = SILVER_DIR / 'jobs.parquet'
DRIVE_SILVER_PATH = PERSIST_ROOT / 'data/silver/jobs.parquet'
ARTIFACT_DIR = PERSIST_ROOT / 'artifacts/skill_timelines'
PARQUET_DIR = ARTIFACT_DIR / 'parquet'
PLOT_DIR = ARTIFACT_DIR / 'plots'
DB_PATH = ARTIFACT_DIR / 'jobs_skills.duckdb'
MANIFEST_PATH = ARTIFACT_DIR / 'run_manifest.json'
BUNDLE_PATH = ARTIFACT_DIR / 'skill_timelines_bundle.zip'

REUSE_DRIVE_SILVER = True
TIME_COLUMN = os.environ.get('VECTORJOBS_TIME_COLUMN') or None
TOP_N_SKILLS = int(os.environ.get('TOP_N_SKILLS', '12'))

QWEN_MODEL_ID = os.environ.get('QWEN_MODEL_ID', 'Qwen/Qwen3-Embedding-0.6B')
QWEN_BATCH_SIZE = int(os.environ.get('QWEN_BATCH_SIZE', '64'))
DOMAIN_CONFIDENCE_THRESHOLD = float(os.environ.get('DOMAIN_CONFIDENCE_THRESHOLD', '0.22'))
DOMAIN_MARGIN_THRESHOLD = float(os.environ.get('DOMAIN_MARGIN_THRESHOLD', '0.03'))

RUN_LLM_ENRICHMENT = os.environ.get('RUN_LLM_ENRICHMENT', '0') == '1'
LLM_MODEL_ID = os.environ.get('LLM_MODEL_ID', 'openai/gpt-oss-20b')
LLM_MAX_NEW_TOKENS = int(os.environ.get('LLM_MAX_NEW_TOKENS', '384'))
LLM_SAMPLE_SIZE = int(os.environ.get('LLM_SAMPLE_SIZE', '250'))

TARGET_DOMAINS = ['health', 'tech', 'construction_industry', 'education', 'sales']
DOMAIN_LABELS = {
    'health': 'Health',
    'tech': 'Tech',
    'construction_industry': 'Construction/Industry',
    'education': 'Education',
    'sales': 'Sales',
}

for path in [SILVER_DIR, ARTIFACT_DIR, PARQUET_DIR, PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

run_manifest = {
    'created_at': datetime.now(tz=timezone.utc).isoformat(),
    'repo_url': REPO_URL,
    'branch': BRANCH,
    'qwen_model_id': QWEN_MODEL_ID,
    'llm_model_id': LLM_MODEL_ID,
    'run_llm_enrichment': RUN_LLM_ENRICHMENT,
    'artifact_dir': str(ARTIFACT_DIR),
}

print('IN_COLAB:', IN_COLAB)
print('REPO_DIR:', REPO_DIR)
print('ARTIFACT_DIR:', ARTIFACT_DIR)
print('BRANCH:', BRANCH)

## 1. Clone and install repo

In [ ]:
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

extra_packages = [
    'kaggle',
    'duckdb',
    'sentence-transformers',
    'transformers',
    'accelerate',
    'bitsandbytes',
    'plotly',
    'seaborn',
    'kaleido',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', *extra_packages], check=True)

import duckdb
import plotly.express as px
import seaborn as sns
from IPython.display import display

run_manifest['install_complete'] = True
run_manifest['repo_dir'] = str(REPO_DIR)

## 2. Get or build data

In [ ]:
def _copy_if_exists(src: Path, dst: Path) -> bool:
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        return True
    return False

if REUSE_DRIVE_SILVER and DRIVE_SILVER_PATH != SILVER_PATH and _copy_if_exists(DRIVE_SILVER_PATH, SILVER_PATH):
    print('Reused silver data from Drive:', DRIVE_SILVER_PATH)
    run_manifest['silver_source'] = 'drive_cache'
elif SILVER_PATH.exists():
    print('Using existing repo silver data:', SILVER_PATH)
    run_manifest['silver_source'] = 'repo_existing'
else:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json = kaggle_dir / 'kaggle.json'

    if IN_COLAB and not kaggle_json.exists():
        from google.colab import files
        print('Upload kaggle.json for dataset arshkon/linkedin-job-postings')
        uploaded = files.upload()
        if 'kaggle.json' not in uploaded:
            raise FileNotFoundError('kaggle.json was not uploaded.')
        Path('kaggle.json').replace(kaggle_json)
        os.chmod(kaggle_json, 0o600)

    if not (RAW_DIR / 'postings.csv').exists():
        subprocess.run([
            'kaggle', 'datasets', 'download',
            '-d', 'arshkon/linkedin-job-postings',
            '-p', str(RAW_DIR),
            '--unzip',
        ], check=True)

    subprocess.run([
        sys.executable, '-m', 'jobsrec.cli', 'build-silver',
        '--input-dir', str(RAW_DIR),
        '--output-dir', str(SILVER_DIR),
    ], check=True)
    run_manifest['silver_source'] = 'kaggle_build'

    if DRIVE_SILVER_PATH != SILVER_PATH:
        DRIVE_SILVER_PATH.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(SILVER_PATH, DRIVE_SILVER_PATH)
        print('Cached silver data to Drive:', DRIVE_SILVER_PATH)

silver = pd.read_parquet(SILVER_PATH)
required = {'job_id', 'job_card_text', 'skills_text'}
missing = required - set(silver.columns)
assert not missing, f'Missing required silver columns: {missing}'
assert len(silver) > 0, 'Silver data is empty.'
print('Silver rows:', len(silver))
print('Silver columns:', list(silver.columns))
run_manifest['silver_path'] = str(SILVER_PATH)
run_manifest['silver_rows'] = int(len(silver))

## 3. Temporal audit

In [ ]:
TIME_CANDIDATES = ['posted_at', 'posting_date', 'created_at', 'listed_time', 'original_listed_time', 'expiry', 'closed_time']

def parse_datetime_series(values: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(values, errors='coerce')
    numeric = pd.to_numeric(values, errors='coerce')
    if numeric.notna().any():
        parsed_years = parsed.dropna().dt.year
        suspicious = parsed_years.empty or parsed_years.median() < 1990
        if suspicious:
            for unit in ['ms', 's']:
                reparsed = pd.to_datetime(numeric, errors='coerce', unit=unit)
                years = reparsed.dropna().dt.year
                if not years.empty and 1990 <= years.median() <= 2100:
                    return reparsed
    return parsed

present_time_columns = [column for column in TIME_CANDIDATES if column in silver.columns]
if not present_time_columns:
    raise ValueError(f'No usable time column found. Tried: {TIME_CANDIDATES}')

time_column = TIME_COLUMN or present_time_columns[0]
if time_column not in silver.columns:
    raise ValueError(f'Configured TIME_COLUMN={time_column!r} is not in silver data.')

jobs = silver.copy()
jobs['posted_at'] = parse_datetime_series(jobs[time_column])
jobs = jobs[jobs['posted_at'].notna()].copy()
jobs['month'] = jobs['posted_at'].dt.to_period('M').astype(str)
jobs['job_card_text'] = jobs['job_card_text'].fillna('').astype(str)
jobs['skills_text'] = jobs['skills_text'].fillna('').astype(str)

assert len(jobs) > 0, f'Time column {time_column} produced no parseable timestamps.'
month_counts = jobs.groupby('month').size().reset_index(name='jobs').sort_values('month')
print('Selected time column:', time_column)
display(month_counts)

if month_counts['month'].nunique() <= 2:
    print('Coverage warning: data spans two or fewer listed months. Treat plots as observed-bucket comparisons, not stable long-term trends.')

run_manifest['time_column'] = time_column
run_manifest['month_coverage'] = month_counts.to_dict(orient='records')
run_manifest['jobs_after_time_filter'] = int(len(jobs))

## 4. Build Qwen embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_cache_path = ARTIFACT_DIR / 'qwen_job_embeddings.npy'
embedding_ids_path = ARTIFACT_DIR / 'qwen_job_embedding_job_ids.npy'

job_ids = jobs['job_id'].to_numpy()
texts = jobs['job_card_text'].fillna('').astype(str).tolist()

def load_sentence_transformer(model_id: str):
    device = 'cuda' if torch is not None and torch.cuda.is_available() else 'cpu'
    try:
        return SentenceTransformer(model_id, device=device, trust_remote_code=True)
    except TypeError:
        return SentenceTransformer(model_id, device=device)

cache_ok = False
if embedding_cache_path.exists() and embedding_ids_path.exists():
    cached_ids = np.load(embedding_ids_path, allow_pickle=False)
    cache_ok = len(cached_ids) == len(job_ids) and np.array_equal(cached_ids, job_ids)

if cache_ok:
    job_embeddings = np.load(embedding_cache_path)
    print('Loaded cached Qwen embeddings:', embedding_cache_path, job_embeddings.shape)
    qwen_model = load_sentence_transformer(QWEN_MODEL_ID)
else:
    qwen_model = load_sentence_transformer(QWEN_MODEL_ID)
    job_embeddings = qwen_model.encode(
        texts,
        batch_size=QWEN_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    job_embeddings = np.asarray(job_embeddings, dtype=np.float32)
    np.save(embedding_cache_path, job_embeddings)
    np.save(embedding_ids_path, job_ids)
    print('Saved Qwen embeddings:', embedding_cache_path, job_embeddings.shape)

run_manifest['qwen_embedding_cache_path'] = str(embedding_cache_path)
run_manifest['qwen_embedding_shape'] = list(job_embeddings.shape)

## 5. Assign job domains

In [ ]:
DOMAIN_CONFIG = {
    'health': {
        'label': 'Health',
        'keywords': ['patient', 'nursing', 'nurse', 'medical', 'clinic', 'health care', 'healthcare', 'health', 'hospital', 'pharmacy', 'physician', 'therapy'],
        'prompt': 'Jobs in health care, hospitals, clinics, nursing, patient care, pharmacy, medical operations, therapy, public health, and clinical support.',
    },
    'tech': {
        'label': 'Tech',
        'keywords': ['software', 'developer', 'engineer', 'information technology', 'technology', 'data', 'python', 'sql', 'cloud', 'machine learning', 'cybersecurity', 'devops'],
        'prompt': 'Jobs in software engineering, data science, analytics, cloud infrastructure, cybersecurity, IT, product engineering, and developer tools.',
    },
    'construction_industry': {
        'label': 'Construction/Industry',
        'keywords': ['construction', 'manufacturing', 'industrial', 'warehouse', 'logistics', 'maintenance', 'machinery', 'site', 'contractor', 'safety', 'electrical', 'mechanical', 'plant', 'operator'],
        'prompt': 'Jobs in construction, industrial operations, manufacturing, logistics, maintenance, plant operations, trades, site safety, and engineering operations.',
    },
    'education': {
        'label': 'Education',
        'keywords': ['education', 'students', 'student', 'teacher', 'teaching', 'school', 'training', 'curriculum', 'classroom', 'academic', 'instructor', 'learning'],
        'prompt': 'Jobs in education, schools, universities, teaching, curriculum, learning programs, academic administration, training, and student services.',
    },
    'sales': {
        'label': 'Sales',
        'keywords': ['sales', 'business development', 'customer service', 'account', 'revenue', 'client', 'crm', 'lead generation', 'retail', 'territory', 'quota'],
        'prompt': 'Jobs in sales, business development, account management, retail sales, customer success, revenue operations, client growth, and CRM workflows.',
    },
}

def keyword_count(text: str, keywords: list[str]) -> int:
    normalized = ' '.join(str(text).lower().replace('/', ' ').replace('|', ' ').replace(';', ' ').split())
    return sum(normalized.count(keyword) for keyword in keywords)

seed_prompts = [DOMAIN_CONFIG[domain]['prompt'] for domain in TARGET_DOMAINS]
seed_embeddings = qwen_model.encode(seed_prompts, normalize_embeddings=True, show_progress_bar=False)
seed_embeddings = np.asarray(seed_embeddings, dtype=np.float32)

similarities = np.matmul(job_embeddings, seed_embeddings.T)
best_idx = similarities.argmax(axis=1)
sorted_sims = np.sort(similarities, axis=1)
best_scores = sorted_sims[:, -1]
second_scores = sorted_sims[:, -2] if similarities.shape[1] > 1 else np.zeros(len(similarities))
margins = best_scores - second_scores

rows = []
for row_pos, (_, row) in enumerate(jobs.iterrows()):
    text = f"{row.get('job_card_text', '')} {row.get('skills_text', '')}"
    embedding_domain = TARGET_DOMAINS[int(best_idx[row_pos])]
    keyword_scores = {domain: keyword_count(text, DOMAIN_CONFIG[domain]['keywords']) for domain in TARGET_DOMAINS}
    keyword_domain, keyword_score = max(keyword_scores.items(), key=lambda item: item[1])

    domain = embedding_domain
    source = 'embedding'
    if keyword_score >= 2 or (keyword_score >= 1 and best_scores[row_pos] < DOMAIN_CONFIDENCE_THRESHOLD):
        domain = keyword_domain
        source = 'keyword_fallback'

    is_uncertain = bool(best_scores[row_pos] < DOMAIN_CONFIDENCE_THRESHOLD or margins[row_pos] < DOMAIN_MARGIN_THRESHOLD)
    rows.append({
        'job_id': row['job_id'],
        'domain': domain,
        'domain_label': DOMAIN_LABELS[domain],
        'domain_score': float(best_scores[row_pos]),
        'domain_margin': float(margins[row_pos]),
        'embedding_domain': embedding_domain,
        'keyword_domain': keyword_domain if keyword_score > 0 else None,
        'keyword_score': int(keyword_score),
        'domain_source': source,
        'is_uncertain': is_uncertain,
    })

job_domains = pd.DataFrame(rows)
job_domains['domain'] = pd.Categorical(job_domains['domain'], categories=TARGET_DOMAINS)
assert list(job_domains['domain'].cat.categories) == TARGET_DOMAINS
assert job_domains['job_id'].nunique() == len(job_domains)

domain_counts = job_domains.groupby('domain', observed=False).size().reset_index(name='jobs')
display(domain_counts)
run_manifest['domain_counts'] = domain_counts.to_dict(orient='records')

## 6. Build skill long table

In [ ]:
def normalize_skill(value: object) -> str | None:
    text = str(value or '').strip()
    text = re.sub(r'\s+', ' ', text)
    text = text.strip(' .;:,|/\\')
    if not text:
        return None
    lowered = text.lower()
    canonical = {
        'sql': 'SQL',
        'python': 'Python',
        'java': 'Java',
        'javascript': 'JavaScript',
        'aws': 'AWS',
        'gcp': 'GCP',
        'azure': 'Azure',
        'c++': 'C++',
        'c#': 'C#',
        'r': 'R',
    }
    return canonical.get(lowered, ' '.join(part.capitalize() for part in lowered.split()))

def split_skills(value: object) -> list[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    parts = re.split(r'[;,|]', str(value))
    normalized = []
    seen = set()
    for part in parts:
        skill = normalize_skill(part)
        if skill and skill.lower() not in seen:
            normalized.append(skill)
            seen.add(skill.lower())
    return normalized

skill_rows = []
for _, row in jobs[['job_id', 'skills_text']].iterrows():
    for skill in split_skills(row['skills_text']):
        skill_rows.append({
            'job_id': row['job_id'],
            'skill': skill,
            'source': 'silver',
            'confidence': 1.0,
        })

job_skills_long = pd.DataFrame(skill_rows, columns=['job_id', 'skill', 'source', 'confidence'])
print('Deterministic skill rows:', len(job_skills_long))
display(job_skills_long.head())
run_manifest['silver_skill_rows'] = int(len(job_skills_long))

## 7. Optional GPT-OSS 20B enrichment

In [ ]:
llm_schema = ['job_id', 'raw_response', 'raw_json', 'validation_status', 'error', 'extracted_skill_count']
llm_skill_enrichment = pd.DataFrame(columns=llm_schema)
llm_skill_rows = []

existing_by_job = (
    job_skills_long.groupby('job_id')['skill']
    .apply(lambda values: {str(value).lower() for value in values})
    .to_dict()
)

candidate_jobs = jobs.merge(job_domains[['job_id', 'is_uncertain']], on='job_id', how='left')
candidate_jobs['_has_skills'] = candidate_jobs['skills_text'].fillna('').astype(str).str.strip().ne('')
candidate_jobs = candidate_jobs[(~candidate_jobs['_has_skills']) | (candidate_jobs['is_uncertain'].fillna(False))]
if LLM_SAMPLE_SIZE > 0:
    candidate_jobs = candidate_jobs.head(LLM_SAMPLE_SIZE)

print('LLM enrichment enabled:', RUN_LLM_ENRICHMENT)
print('LLM candidate rows:', len(candidate_jobs))

if RUN_LLM_ENRICHMENT and len(candidate_jobs) > 0:
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, trust_remote_code=True)
        quantization_config = None
        model_kwargs = {'device_map': 'auto', 'trust_remote_code': True}
        if torch.cuda.is_available():
            quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
            model_kwargs['quantization_config'] = quantization_config
        model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_ID, **model_kwargs)

        def extract_json(text: str) -> dict:
            start = text.find('{')
            end = text.rfind('}')
            if start == -1 or end == -1 or end <= start:
                raise ValueError('No JSON object found in model response.')
            return json.loads(text[start:end + 1])

        def prompt_for_job(job_text: str) -> str:
            return (
                'Extract concrete skills from this job posting. Return strict JSON only with this schema: '
                '{"skills":[{"name":"...","type":"technical|domain|soft|tool|certification","evidence":"..."}]}\n\n'
                f'Job posting:\n{job_text[:5000]}'
            )

        enrichment_records = []
        for _, row in candidate_jobs.iterrows():
            prompt = prompt_for_job(row['job_card_text'])
            inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
            outputs = model.generate(**inputs, max_new_tokens=LLM_MAX_NEW_TOKENS, do_sample=False)
            decoded = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            try:
                parsed = extract_json(decoded)
                skills = parsed.get('skills', [])
                if not isinstance(skills, list):
                    raise ValueError('skills is not a list')
                valid_count = 0
                for item in skills:
                    if not isinstance(item, dict):
                        continue
                    skill = normalize_skill(item.get('name'))
                    evidence = str(item.get('evidence', '')).strip()
                    skill_type = str(item.get('type', '')).strip().lower()
                    if not skill or len(skill) > 80 or skill_type not in {'technical', 'domain', 'soft', 'tool', 'certification'}:
                        continue
                    known = existing_by_job.setdefault(row['job_id'], set())
                    if skill.lower() in known:
                        continue
                    known.add(skill.lower())
                    valid_count += 1
                    llm_skill_rows.append({
                        'job_id': row['job_id'],
                        'skill': skill,
                        'source': 'llm',
                        'confidence': 0.65,
                    })
                enrichment_records.append({
                    'job_id': row['job_id'],
                    'raw_response': decoded,
                    'raw_json': json.dumps(parsed),
                    'validation_status': 'valid',
                    'error': None,
                    'extracted_skill_count': valid_count,
                })
            except Exception as exc:
                enrichment_records.append({
                    'job_id': row['job_id'],
                    'raw_response': decoded,
                    'raw_json': None,
                    'validation_status': 'invalid',
                    'error': str(exc),
                    'extracted_skill_count': 0,
                })

        llm_skill_enrichment = pd.DataFrame(enrichment_records, columns=llm_schema)
        run_manifest['llm_enrichment_status'] = 'completed'
    except Exception as exc:
        run_manifest['llm_enrichment_status'] = 'skipped_model_load_failed'
        run_manifest['llm_enrichment_error'] = str(exc)
        print('LLM enrichment skipped after model/load/generation failure:', exc)
else:
    run_manifest['llm_enrichment_status'] = 'disabled_or_no_candidates'

if llm_skill_rows:
    job_skills_long = pd.concat([job_skills_long, pd.DataFrame(llm_skill_rows)], ignore_index=True)
    job_skills_long = job_skills_long.drop_duplicates(subset=['job_id', 'skill', 'source']).reset_index(drop=True)

run_manifest['llm_skill_rows'] = int(len(llm_skill_rows))
run_manifest['total_skill_rows'] = int(len(job_skills_long))
print('LLM skill rows added:', len(llm_skill_rows))

## 8. Persist DuckDB and Parquet

In [ ]:
jobs_table_columns = list(dict.fromkeys([
    column for column in [
        'job_id', 'title', 'description', 'job_card_text', 'skills_text', 'posted_at', 'month',
        'formatted_work_type', 'formatted_experience_level', 'work_type', 'location',
        'remote_allowed', 'normalized_salary', 'min_salary', 'max_salary', 'med_salary',
        'pay_period', 'currency', 'company_id', time_column,
    ] if column in jobs.columns
]))
jobs_table = jobs[jobs_table_columns].copy()

skill_domain = (
    job_skills_long
    .merge(job_domains[['job_id', 'domain', 'domain_label']], on='job_id', how='left')
    .merge(jobs[['job_id', 'month']], on='job_id', how='left')
)

domain_job_counts = (
    job_domains.merge(jobs[['job_id', 'month']], on='job_id', how='left')
    .groupby(['domain', 'month'], observed=False)['job_id']
    .nunique()
    .reset_index(name='job_count')
)
skill_counts = (
    skill_domain.dropna(subset=['skill', 'month'])
    .groupby(['domain', 'month', 'skill'], observed=True)['job_id']
    .nunique()
    .reset_index(name='skill_job_count')
)
domain_skill_monthly = skill_counts.merge(domain_job_counts, on=['domain', 'month'], how='left')
domain_skill_monthly['share_pct'] = np.where(
    domain_skill_monthly['job_count'] > 0,
    100.0 * domain_skill_monthly['skill_job_count'] / domain_skill_monthly['job_count'],
    0.0,
)
domain_skill_monthly['share_pct'] = domain_skill_monthly['share_pct'].clip(lower=0, upper=100)

assert domain_skill_monthly['share_pct'].between(0, 100).all()

con = duckdb.connect(str(DB_PATH))
tables = {
    'jobs': jobs_table,
    'job_skills_long': job_skills_long,
    'job_domains': job_domains.copy(),
    'domain_skill_monthly': domain_skill_monthly,
    'llm_skill_enrichment': llm_skill_enrichment,
}

PARQUET_DIR.mkdir(parents=True, exist_ok=True)
for table_name, frame in tables.items():
    frame_to_write = frame.copy()
    for column in frame_to_write.select_dtypes(include='category').columns:
        frame_to_write[column] = frame_to_write[column].astype(str)
    con.register('frame_to_write', frame_to_write)
    con.execute(f'CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM frame_to_write')
    con.execute(f"COPY {table_name} TO '{(PARQUET_DIR / (table_name + '.parquet')).as_posix()}' (FORMAT PARQUET)")
    con.unregister('frame_to_write')

silver_count = len(silver)
duckdb_jobs_count = con.execute('SELECT COUNT(*) FROM jobs').fetchone()[0]
assert duckdb_jobs_count == len(jobs_table)
print('DuckDB path:', DB_PATH)
print('DuckDB jobs rows:', duckdb_jobs_count, 'Silver rows:', silver_count, 'After time filter:', len(jobs_table))

display(domain_job_counts.sort_values(['domain', 'month']))
run_manifest['duckdb_path'] = str(DB_PATH)
run_manifest['parquet_dir'] = str(PARQUET_DIR)
run_manifest['duckdb_jobs_count'] = int(duckdb_jobs_count)

## 9. Plot skill timelines

In [ ]:
plot_records = []

for domain in TARGET_DOMAINS:
    domain_df = domain_skill_monthly[domain_skill_monthly['domain'].astype(str) == domain].copy()
    html_path = PLOT_DIR / f'skill_timeline_{domain}.html'
    png_path = PLOT_DIR / f'skill_timeline_{domain}.png'
    note_path = PLOT_DIR / f'skill_timeline_{domain}_insufficient_data.txt'

    if domain_df.empty or domain_df['month'].nunique() == 0:
        note_path.write_text(f'No skill timeline data for {DOMAIN_LABELS[domain]}.\n', encoding='utf-8')
        plot_records.append({'domain': domain, 'html': None, 'png': None, 'note': str(note_path), 'status': 'insufficient_data'})
        continue

    top_skills = (
        domain_df.groupby('skill')['skill_job_count']
        .sum()
        .sort_values(ascending=False)
        .head(TOP_N_SKILLS)
        .index
        .tolist()
    )
    plot_df = domain_df[domain_df['skill'].isin(top_skills)].sort_values(['month', 'skill'])

    fig = px.line(
        plot_df,
        x='month',
        y='share_pct',
        color='skill',
        markers=True,
        title=f"{DOMAIN_LABELS[domain]}: Skill share by month",
        labels={'month': 'Month', 'share_pct': 'Share of domain jobs requiring skill (%)', 'skill': 'Skill'},
    )
    fig.update_layout(legend_title_text='Skill', yaxis_range=[0, min(100, max(10, float(plot_df['share_pct'].max()) * 1.15))])
    fig.write_html(str(html_path), include_plotlyjs='cdn')

    png_status = 'saved'
    try:
        fig.write_image(str(png_path), scale=2)
    except Exception as exc:
        png_status = f'png_failed: {exc}'
        note_path.write_text(f'Interactive HTML was saved, but PNG export failed: {exc}\n', encoding='utf-8')

    plot_records.append({
        'domain': domain,
        'html': str(html_path),
        'png': str(png_path) if png_path.exists() else None,
        'note': str(note_path) if note_path.exists() else None,
        'status': png_status,
    })

plot_outputs = pd.DataFrame(plot_records)
display(plot_outputs)

for record in plot_records:
    assert record['html'] or record['note'], f"Missing plot or insufficient-data note for {record['domain']}"

run_manifest['plot_outputs'] = plot_records

## 10. Export bundle

In [ ]:
run_manifest['completed_at'] = datetime.now(tz=timezone.utc).isoformat()
MANIFEST_PATH.write_text(json.dumps(run_manifest, indent=2, default=str), encoding='utf-8')

bundle_members = []
with zipfile.ZipFile(BUNDLE_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in [MANIFEST_PATH, DB_PATH]:
        if path.exists():
            zf.write(path, arcname=path.relative_to(ARTIFACT_DIR))
            bundle_members.append(str(path))
    for folder in [PARQUET_DIR, PLOT_DIR]:
        for path in folder.rglob('*'):
            if path.is_file():
                zf.write(path, arcname=path.relative_to(ARTIFACT_DIR))
                bundle_members.append(str(path))

print('Bundle:', BUNDLE_PATH)
print('Bundle members:', len(bundle_members))

if IN_COLAB:
    from google.colab import files
    files.download(str(BUNDLE_PATH))

## Validation summary

In [ ]:
assert {'job_id', 'job_card_text', 'skills_text'}.issubset(silver.columns)
posted_months = jobs['month'].nunique()
assert posted_months > 0
assert list(job_domains['domain'].cat.categories) == TARGET_DOMAINS
assert domain_skill_monthly.empty or domain_skill_monthly['share_pct'].between(0, 100).all()
assert all(record['html'] or record['note'] for record in plot_records)
assert con.execute('SELECT COUNT(*) FROM jobs').fetchone()[0] == len(jobs_table)

print('Validation passed.')
print('Months available:', posted_months)
print('Domain row counts:')
display(domain_counts)
print('Artifacts:', ARTIFACT_DIR)